In [59]:
import numpy as np 
import modern_robotics as mr 
import math as mt

In [60]:
def calculate_space_jacobian():
    # 1. Home Configuration M
    M = np.array([
        [-1, 0, 0, 0],
        [0, 0, 1, 3],
        [0, 1, 0, 2],
        [0, 0, 0, 1]
    ])

    # 2. Space Screw Axes Slist
    S1 = [0, 0, 1,  0, 0, 0]
    S2 = [1, 0, 0,  0, 2, 0]
    S3 = [0, 0, 0,  0, 1, 0]
    Slist = np.array([S1, S2, S3]).T

    # 3. Current Configuration
    # theta = (90 deg, 90 deg, 1 meter) -> Convert deg to rad
    thetalist = np.array([np.pi/2, np.pi/2, 1.0])

    # 4. Calculate Body Screw Axes (Blist) from Space Screws (Slist)
    # B_i = Ad(M^-1) * S_i
    M_inv = np.linalg.inv(M)
    Blist = mr.Adjoint(M_inv) @ Slist

    # 5. Calculate Body Jacobian
    Js = mr.JacobianSpace(Slist, thetalist)
    
    return Js

# Run the calculation and print the matrix rounded to 4 decimal places
Js = calculate_space_jacobian()

print("Space Jacobian (Js):")
print(np.round(Js, 4))

Space Jacobian (Js):
[[ 0.  0.  0.]
 [ 0.  1.  0.]
 [ 1.  0.  0.]
 [ 0. -2. -0.]
 [ 0.  0.  0.]
 [ 0.  0.  1.]]


In [61]:
def calculate_body_jacobian():
    # 1. Home Configuration M
    M = np.array([
        [-1,  0,  0,  0],
        [0,  0,  1,  3],
        [0,  1,  0,  2],
        [0,  0,  0,  1]
    ])

    # 2. Space Screw Axes Slist
    S1 = [ 0,  1,  0,  3,  0,  0]
    S2 = [-1,  0,  0,  0,  3,  0]
    S3 = [ 0,  0,  0,  0,  0,  1]
    
    # Transpose to make each screw axis a column (6x3 matrix)
    Slist = np.array([S1, S2, S3]).T

    # 3. Current Configuration
    # theta = (90 deg, 90 deg, 1 meter) -> Convert to rad
    thetalist = np.array([np.pi/2, np.pi/2, 1.0])

    # 4. Calculate Body Screw Axes (Blist) from Space Screws (Slist)
    # B_i = Ad(M^-1) * S_i
    M_inv = mr.TransInv(M)
    Blist = mr.Adjoint(M_inv) @ Slist

    # 5. Calculate Body Jacobian
    Jb = mr.JacobianBody(Blist, thetalist)
    
    return Jb

# Run the calculation and print the matrix rounded to 4 decimal places
Jb = calculate_body_jacobian()

print("Body Jacobian (Jb):")
print(np.round(Jb, 4))

Body Jacobian (Jb):
[[ 0.  1.  0.]
 [ 1.  0.  0.]
 [ 0.  0.  0.]
 [ 3.  0.  0.]
 [ 0. -3.  1.]
 [ 0.  6.  0.]]


In [62]:
J_b = np.array([
    [0, -1, 0, 0, -1, 0, 0],
    [0, 0, 1, 0, 0, 1, 0],
    [1, 0, 0, 1, 0, 0, 1],
    [-0.105, 0, 0.006, -0.045, 0, 0.006, 0],
    [-0.889, 0.006, 0, -0.844, 0.006, 0, 0],
    [0, -0.105, 0.889, 0, 0, 0, 0]
])

# Extract the linear velocity portion Jv (bottom 3 rows)
J_v = J_b[3:, :]

# Perform Singular Value Decomposition (SVD)
U, S, Vt = np.linalg.svd(J_v)

# Extract longest principal semi-axis direction (first column of U)
longest_axis_direction = U[:, 0]

# Print results
print("Linear Velocity Jacobian (Jv):")
print(J_v)
print("\nLengths of principal semi-axes (Singular values):")
print(np.round(S, 4))
print("\nDirections of principal semi-axes (Left singular vectors, columns of U):")
print(np.round(U, 4))
print("\nDirection of the longest principal semi-axis (Unit vector):")
print(np.round(longest_axis_direction, 2))


Linear Velocity Jacobian (Jv):
[[-0.105  0.     0.006 -0.045  0.     0.006  0.   ]
 [-0.889  0.006  0.    -0.844  0.006  0.     0.   ]
 [ 0.    -0.105  0.889  0.     0.     0.     0.   ]]

Lengths of principal semi-axes (Singular values):
[1.2305 0.8952 0.04  ]

Directions of principal semi-axes (Left singular vectors, columns of U):
[[-8.720e-02  6.700e-03 -9.962e-01]
 [-9.962e-01 -4.000e-04  8.720e-02]
 [ 2.000e-04  1.000e+00  6.700e-03]]

Direction of the longest principal semi-axis (Unit vector):
[-0.09 -1.    0.  ]


[-1.4142,-1.4142,0]
[30, 20, 10, 20]
